# Pizza Sales Analysis — Data Preparation & EDA


## Project Introduction

### In the competitive food and restaurant industry, understanding customer ordering behavior and sales trends is essential for maximizing revenue and improving business decisions. This project focuses on analyzing a pizza restaurant’s transactional sales data using SQL and Python within Jupyter Notebook.

### The dataset contains historical order information, including order time, pizza types, sizes, quantities, and prices. However, the data is stored across multiple tables, which requires data cleaning, transformation, and joining before meaningful analysis can be performed.



## Objective
### The goal of this notebook is to transform raw datasets into a clean and analysis-ready final dataset that will later be used for SQL business analysis and visualization.

## Importing Libraries

In [ ]:
import pandas as pd
import os
from sqlalchemy import create_engine
import logging
import time
import warnings
warnings.filterwarnings('ignore')
logging.basicConfig(
    filename = 'logs/pizza_sales.db.log',
    level = logging.DEBUG,
    format = '%(asctime)s - %(levelname)s - %(message)s',
    filemode = 'a'
)

engine = create_engine('sqlite:///pizza_sales_analysis.db')

## Connecting to SQL Database and Loading CSV Data into DataFrame

In [ ]:
def ingest_db(df,table_name,engine):
    ''' this function will ingest the dataframe into database table'''
    df.to_sql(table_name,con = engine,if_exists = 'replace',index = False)
def read_data():
    ''' this function will load CSVs as dataframe and ingest into db'''
    start = time.time()
    
    for i in os.listdir('pizza_sales_data'):
        if '.csv' in i:
            df = pd.read_csv('pizza_sales_data/'+i, encoding='latin-1')
            logging.info(f'ingest:{i} in db')
            ingest_db(df,i[:-4],engine)
    
    end = time.time()
    total = (end - start)/60
    logging.info('ingestion complete')
    logging.info(f'total time taken:{total} minute')

if __name__ == '__main__':
    read_data()


In [71]:
# Verifying Tables Created in the Database
df = pd.read_sql_query('select name from sqlite_master where type = "table"',engine)

In [72]:
# Exploring and Understanding the Raw Tables
for i in df['name']:
    print('*'*50,f'{i}','*'*50)
    print('count records:',pd.read_sql_query(f'select count(*) as count from {i}',engine)['count'].values[0])
    display(pd.read_sql_query(f'select * from {i} limit 5',engine))

************************************************** pizza **************************************************
count records: 48620


,order_id,name,category,size,price,quantity,revenue,month_name,hour,weekday
0,1,The Hawaiian Pizza,Classic,M,13.25,1,13.25,January,11,Thursday
1,2,The Classic Deluxe Pizza,Classic,M,16.00,1,16.00,January,11,Thursday
2,2,The Five Cheese Pizza,Veggie,L,18.50,1,18.50,January,11,Thursday
3,2,The Italian Supreme Pizza,Supreme,L,20.75,1,20.75,January,11,Thursday
4,2,The Mexicana Pizza,Veggie,M,16.00,1,16.00,January,11,Thursday


************************************************** orders **************************************************
count records: 21350


,order_id,date,time
0,1,2015-01-01,11:38:36
1,2,2015-01-01,11:57:40
2,3,2015-01-01,12:12:28
3,4,2015-01-01,12:16:31
4,5,2015-01-01,12:21:30


************************************************** order_details **************************************************
count records: 48620


,order_details_id,order_id,pizza_id,quantity
0,1,1,hawaiian_m,1
1,2,2,classic_dlx_m,1
2,3,2,five_cheese_l,1
3,4,2,ital_supr_l,1
4,5,2,mexicana_m,1


************************************************** pizzas **************************************************
count records: 96


,pizza_id,pizza_type_id,size,price
0,bbq_ckn_s,bbq_ckn,S,12.75
1,bbq_ckn_m,bbq_ckn,M,16.75
2,bbq_ckn_l,bbq_ckn,L,20.75
3,cali_ckn_s,cali_ckn,S,12.75
4,cali_ckn_m,cali_ckn,M,16.75


************************************************** pizza_types **************************************************
count records: 32


,pizza_type_id,name,category,ingredients
0,bbq_ckn,The Barbecue Chicken Pizza,Chicken,"Barbecued Chicken, Red Peppers, Green Peppers,..."
1,cali_ckn,The California Chicken Pizza,Chicken,"Chicken, Artichoke, Spinach, Garlic, Jalapeno ..."
2,ckn_alfredo,The Chicken Alfredo Pizza,Chicken,"Chicken, Red Onions, Red Peppers, Mushrooms, A..."
3,ckn_pesto,The Chicken Pesto Pizza,Chicken,"Chicken, Tomatoes, Red Peppers, Spinach, Garli..."
4,southw_ckn,The Southwest Chicken Pizza,Chicken,"Chicken, Tomatoes, Red Peppers, Red Onions, Ja..."


## Combining Required Data into a Single Unified Table

In [73]:
pd.read_sql_query("""
    select o.order_id,
    o.date,o.time,
    sum(od.quantity) as quantity
    from order_details od join
    orders o 
    on od.order_id = o.order_id
    group by o.order_id,o.date,o.time""",conn)

,order_id,date,time,quantity
0,1,2015-01-01,11:38:36,1
1,2,2015-01-01,11:57:40,5
2,3,2015-01-01,12:12:28,2
3,4,2015-01-01,12:16:31,1
4,5,2015-01-01,12:21:30,1
...,...,...,...,...
21345,21346,2015-12-31,20:51:07,4
21346,21347,2015-12-31,21:14:37,4
21347,21348,2015-12-31,21:23:10,3
21348,21349,2015-12-31,22:09:54,1


In [74]:
pd.read_sql_query("""
    select pt.name,
    pt.category,p.size,
    sum(p.price) as price
    from pizza_types pt join
    pizzas p 
    on pt.pizza_type_id = p.pizza_type_id
    group by pt.name""",conn)

,name,category,size,price
0,The Barbecue Chicken Pizza,Chicken,L,50.25
1,The Big Meat Pizza,Classic,L,48.50
2,The Brie Carre Pizza,Supreme,S,23.65
3,The Calabrese Pizza,Supreme,L,48.75
4,The California Chicken Pizza,Chicken,L,50.25
5,The Chicken Alfredo Pizza,Chicken,L,50.25
6,The Chicken Pesto Pizza,Chicken,L,50.25
7,The Classic Deluxe Pizza,Classic,L,48.50
8,The Five Cheese Pizza,Veggie,L,46.50
9,The Four Cheese Pizza,Veggie,L,44.45


In [75]:
#  SQL Query for Creating the  Combined Table
pizza = pd.read_sql_query("""
    select o.order_id,
    pt.name,pt.category,p.size,
    p.price ,
    od.quantity ,
    (p.price*od.quantity) as revenue,
    o.date,o.time
    from pizza_types pt join pizzas p 
    on pt.pizza_type_id = p.pizza_type_id
    join order_details od 
    on p.pizza_id = od.pizza_id
    join orders o
    on od.order_id = o.order_id
    group by o.order_id,pt.name,pt.category,p.size,o.date,o.time""",conn)

## Exploring the Combined Table

In [76]:
# dataset info
pizza.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48620 entries, 0 to 48619
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   order_id  48620 non-null  int64  
 1   name      48620 non-null  object 
 2   category  48620 non-null  object 
 3   size      48620 non-null  object 
 4   price     48620 non-null  float64
 5   quantity  48620 non-null  int64  
 6   revenue   48620 non-null  float64
 7   date      48620 non-null  object 
 8   time      48620 non-null  object 
dtypes: float64(2), int64(2), object(5)
memory usage: 3.3+ MB


In [77]:
# check for missing values
pizza.isnull().sum()

order_id    0
name        0
category    0
size        0
price       0
quantity    0
revenue     0
date        0
time        0
dtype: int64

In [78]:
# Distribution of Dataset Variables
for i in pizza.columns:
    if pizza[i].nunique()<20:
        print(pizza[i].value_counts())

category
Classic    14579
Supreme    11777
Veggie     11449
Chicken    10815
Name: count, dtype: int64
size
L      18526
M      15385
S      14137
XL       544
XXL       28
Name: count, dtype: int64
quantity
1    47693
2      903
3       21
4        3
Name: count, dtype: int64


In [79]:
pizza.columns

Index(['order_id', 'name', 'category', 'size', 'price', 'quantity', 'revenue',
       'date', 'time'],
      dtype='object')

In [80]:
# datatypes of the data
pizza.dtypes

order_id      int64
name         object
category     object
size         object
price       float64
quantity      int64
revenue     float64
date         object
time         object
dtype: object

In [81]:
# converting into date format
pizza['date'] = pd.to_datetime(pizza['date'])

In [82]:
# converting into time format
pizza['time'] = pd.to_datetime(pizza['time'],format = '%H:%M:%S')

In [83]:
# dataset info
pizza.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48620 entries, 0 to 48619
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   order_id  48620 non-null  int64         
 1   name      48620 non-null  object        
 2   category  48620 non-null  object        
 3   size      48620 non-null  object        
 4   price     48620 non-null  float64       
 5   quantity  48620 non-null  int64         
 6   revenue   48620 non-null  float64       
 7   date      48620 non-null  datetime64[ns]
 8   time      48620 non-null  datetime64[ns]
dtypes: datetime64[ns](2), float64(2), int64(2), object(3)
memory usage: 3.3+ MB


In [84]:
# extracting month_name from date column
pizza['month_name'] = pizza['date'].dt.month_name()

In [85]:
# extracting hour from time column
pizza['hour'] = pizza['time'].dt.hour

In [86]:
pizza

,order_id,name,category,size,price,quantity,revenue,date,time,month_name,hour
0,1,The Hawaiian Pizza,Classic,M,13.25,1,13.25,2015-01-01,1900-01-01 11:38:36,January,11
1,2,The Classic Deluxe Pizza,Classic,M,16.00,1,16.00,2015-01-01,1900-01-01 11:57:40,January,11
2,2,The Five Cheese Pizza,Veggie,L,18.50,1,18.50,2015-01-01,1900-01-01 11:57:40,January,11
3,2,The Italian Supreme Pizza,Supreme,L,20.75,1,20.75,2015-01-01,1900-01-01 11:57:40,January,11
4,2,The Mexicana Pizza,Veggie,M,16.00,1,16.00,2015-01-01,1900-01-01 11:57:40,January,11
...,...,...,...,...,...,...,...,...,...,...,...
48615,21348,The Chicken Alfredo Pizza,Chicken,M,16.75,1,16.75,2015-12-31,1900-01-01 21:23:10,December,21
48616,21348,The Four Cheese Pizza,Veggie,L,17.95,1,17.95,2015-12-31,1900-01-01 21:23:10,December,21
48617,21348,The Napolitana Pizza,Classic,S,12.00,1,12.00,2015-12-31,1900-01-01 21:23:10,December,21
48618,21349,The Mexicana Pizza,Veggie,L,20.25,1,20.25,2015-12-31,1900-01-01 22:09:54,December,22


In [87]:
# extracting weekdays from the Dataset
pizza['weekday'] = pizza['date'].dt.day_name()

In [88]:
# Removing Unnecessary Columns from the Dataset
pizza = pizza.drop(columns = ['date','time'],axis = 1)

In [89]:
pizza

,order_id,name,category,size,price,quantity,revenue,month_name,hour,weekday
0,1,The Hawaiian Pizza,Classic,M,13.25,1,13.25,January,11,Thursday
1,2,The Classic Deluxe Pizza,Classic,M,16.00,1,16.00,January,11,Thursday
2,2,The Five Cheese Pizza,Veggie,L,18.50,1,18.50,January,11,Thursday
3,2,The Italian Supreme Pizza,Supreme,L,20.75,1,20.75,January,11,Thursday
4,2,The Mexicana Pizza,Veggie,M,16.00,1,16.00,January,11,Thursday
...,...,...,...,...,...,...,...,...,...,...
48615,21348,The Chicken Alfredo Pizza,Chicken,M,16.75,1,16.75,December,21,Thursday
48616,21348,The Four Cheese Pizza,Veggie,L,17.95,1,17.95,December,21,Thursday
48617,21348,The Napolitana Pizza,Classic,S,12.00,1,12.00,December,21,Thursday
48618,21349,The Mexicana Pizza,Veggie,L,20.25,1,20.25,December,22,Thursday


In [95]:
conn = engine.raw_connection()
cursor = conn.cursor()

In [96]:
# creating final talbe
cursor.execute("""create table pizza(
    order_id int,
    name varchar(100),
    category  varchar(100),
    size varchar(100),
    price decimal(10,2),
    quantity int,
    revenue decimal(10,2),
    hour int,
    month_name varchar(100),
    weekday varchar(100),
    primary key (order_id)
    );
    """)

In [92]:
# Saving DataFrame to SQL Table
pizza.to_sql('pizza',conn,if_exists= 'replace',index = False)

48620

In [91]:
# Reading Data from SQL Table into Pandas DataFrame
pd.read_sql_query('select * from pizza',conn)


,order_id,name,category,size,price,quantity,revenue,month_name,hour,weekday
0,1,The Hawaiian Pizza,Classic,M,13.25,1,13.25,January,11,Thursday
1,2,The Classic Deluxe Pizza,Classic,M,16.00,1,16.00,January,11,Thursday
2,2,The Five Cheese Pizza,Veggie,L,18.50,1,18.50,January,11,Thursday
3,2,The Italian Supreme Pizza,Supreme,L,20.75,1,20.75,January,11,Thursday
4,2,The Mexicana Pizza,Veggie,M,16.00,1,16.00,January,11,Thursday
...,...,...,...,...,...,...,...,...,...,...
48615,21348,The Chicken Alfredo Pizza,Chicken,M,16.75,1,16.75,December,21,Thursday
48616,21348,The Four Cheese Pizza,Veggie,L,17.95,1,17.95,December,21,Thursday
48617,21348,The Napolitana Pizza,Classic,S,12.00,1,12.00,December,21,Thursday
48618,21349,The Mexicana Pizza,Veggie,L,20.25,1,20.25,December,22,Thursday


In [33]:
pd.read_sql_query('select * from pizza',engine)

,order_id,name,category,size,quantity,revenue,month_name,hour,weekday
0,1,The Hawaiian Pizza,Classic,M,1,13.25,January,11,Thursday
1,2,The Classic Deluxe Pizza,Classic,M,1,16.00,January,11,Thursday
2,2,The Five Cheese Pizza,Veggie,L,1,18.50,January,11,Thursday
3,2,The Italian Supreme Pizza,Supreme,L,1,20.75,January,11,Thursday
4,2,The Mexicana Pizza,Veggie,M,1,16.00,January,11,Thursday
...,...,...,...,...,...,...,...,...,...
48615,21348,The Chicken Alfredo Pizza,Chicken,M,1,16.75,December,21,Thursday
48616,21348,The Four Cheese Pizza,Veggie,L,1,17.95,December,21,Thursday
48617,21348,The Napolitana Pizza,Classic,S,1,12.00,December,21,Thursday
48618,21349,The Mexicana Pizza,Veggie,L,1,20.25,December,22,Thursday
